# Prompt Engineering Lab


# Part 1: Starting Simple - Create Initial Prompts
## 0. Setup


In [1]:
import os
from openai import OpenAI
import json
from collections import Counter
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Set default model
MODEL = "gpt-4o-mini"

print("✅ Setup complete! OpenAI client initialized.")

✅ Setup complete! OpenAI client initialized.


## Step 2: Create Initial Prompts


In [2]:
def call_openai(prompt, system_message="You are a helpful assistant.", temperature=0.7, max_tokens=None, seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }
        
        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

In [3]:
# Initial simple prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""
 
# Test it once
result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

# Initial simple prompt for product description
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""
 
# Test it once
result = call_openai(product_prompt_v1)
print("Product Description Result:")
print(result)

# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""
 
# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Sentiment Analysis Result:
The customer message can be classified as positive feedback.
Product Description Result:
**Product Description: Wireless Precision Mouse - $29.99**

Upgrade your workspace with our Wireless Precision Mouse, designed for both comfort and functionality. Priced at just $29.99, this sleek and modern mouse combines cutting-edge technology with an ergonomic design, making it the perfect companion for your laptop or desktop.

**Key Features:**

- **Wireless Freedom:** Say goodbye to tangled cords! Our advanced 2.4GHz wireless technology ensures a stable connection with a range of up to 33 feet, allowing you to move freely without restrictions.

- **Ergonomic Design:** Crafted with comfort in mind, the contoured shape fits perfectly in your hand, reducing strain during long hours of use. Ideal for both right and left-handed users.

- **High Precision Tracking:** Equipped with a high-resolution optical sensor, this mouse offers smooth tracking on various surfaces, ens

# Part 2: Diagnosing Failures - Systematic Testing


In [ ]:
def test_prompt(prompt, iterations=5):
    """
    Runs a prompt multiple times to check for consistency.
    """
    results = []
    for i in range(iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        results.append(response.choices[0].message.content.strip())
    
    # Print results for analysis
    for idx, res in enumerate(results):
        print(f"Run {idx+1}: {res}\n")
    return results

## Step 3: Run Prompts 5 Times

In [ ]:

# Run the same prompt 5 times with temperature=0.7
num_runs=5
print("=== TESTING SENTIMENT PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(sentiment_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)
    
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(product_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(extraction_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===

Running 5 times...
Run 1: The customer message can be classified as **Positive Feedback** or **Satisfaction**.

Run 2: This customer message can be classified as "Positive Feedback" or "Customer Appreciation."

Run 3: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 4: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 5: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

📊 Analysis:
Total unique responses: 4 out of 5
Average length: 90 characters
--------------------------------------------------------------------------------
=== TESTING PRODUCT DESCRIPTION PROMPT ===

Running 5 times...
Run 1: **Product Description: Wireless Ergonomic Mouse - $29.99**

Elevate your computing experience with our Wireless Ergonomic Mouse, designed for comfort, precision, and convenience. Priced at just $29.99, this sleek and stylish

## Analysis report for running prompt 5 times
### Are responses consistent in format?
No. 
- Sentiment: The model switches between (**Positive Feedback**) and quotation marks ("Positive Feedback")
- Product Description: While they all default to a paragraph-and-bullet format, the hierarchy, layout structure, and header styles vary.
- Data Extraction: The bullet points fluctuate between placing colons inside the bolding (**Order Number:**), outside the bolding (**Order Number**: ), or using no bolding at all. The extraction prompt kept a consistent bullet-list format but varied the field labels used ("Order Number" vs. "Item Ordered" vs. "Order Item Number").

### Are responses consistent in content?
- Sentiment: Yes, highly consistent — every run classified the message as positive, just with slightly different secondary label wording.
- Product description: No — the content is completely inconsistent. Each run invented different specifics (battery life claims, wireless range, feature lists like "quiet click technology" or "DPI adjustment" appearing in some runs and not others). Because the prompt had no specifications, the model hallucinated different brand names.
- Extraction: Yes, highly consistent — the actual values (#12345, March 15th, Fast delivery, Damaged packaging) were identical across all 5 runs. Only the labels naming those values changed.

### What types of variations do you see?
2 distinct categories:
1. Cosmetic/surface variation — different wording, bold vs. quoted text, different lead-in phrases ("Here is" vs. "Here's"), different field labels. This showed up in all three prompts.
2. Substantive/content variation — actual facts or details changing between runs. This was essentially absent in the sentiment and extraction prompts (the core answer never changed) but significant in the product description prompt, where the model generated different invented specs and even different product names each time.

## Step 4: Run Prompts 10 Times

In [10]:
# Run the same prompt 10 times with temperature=0.7
num_runs=10
print("=== TESTING SENTIMENT PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(sentiment_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)
    
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(product_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(extraction_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===

Running 10 times...
Run 1: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 2: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 3: This customer message can be classified as **positive feedback** or **customer satisfaction**.

Run 4: The customer message can be classified as positive feedback or a positive review.

Run 5: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 6: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 7: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 8: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 9: This customer message can be classified as "Positive Feedback" or "Satisfaction."

Run 10: This customer message can be classified as **

## After running prompts 10 times
The responses are inconsistent in formatting, content and varied in syntax/cosmetics, content. 
A new failure pattern in the sentiment prompt — Run 4 dropped the "Customer Satisfaction" label outright, which never happened in the 5-run test. This confirms the expected outcome: more runs surfaced a rare failure mode that a 5-run sample didn't catch.

## Step 5: Run Prompts 15 Times and Create Failure Analysis

In [11]:
# Run the same prompt 15 times with temperature=0.7
num_runs=15
print("=== TESTING SENTIMENT PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(sentiment_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)
    
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(product_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(extraction_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===

Running 15 times...
Run 1: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 2: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."

Run 3: The customer message can be classified as "Positive Feedback" or "Satisfaction."

Run 4: This customer message can be classified as positive feedback or a customer compliment.

Run 5: This customer message can be classified as positive feedback or a satisfied customer review.

Run 6: This customer message can be classified as **Positive Feedback** or **Satisfaction**.

Run 7: The customer message can be classified as **positive feedback** or **customer satisfaction**.

Run 8: The customer message can be classified as positive feedback or a positive review.

Run 9: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

Run 10: The customer message can be classified as **Positive Feedback*

## Failure Analysis - 15-Run Test (All Three Prompts)

## Summary Table

| Prompt | Unique Responses | Consistency (exact-match) | Avg. Length | Content Reliability |
|---|---|---|---|---|
| Sentiment classification | 12 / 15 (80% unique) | 20% - only 3 runs matched another run verbatim | 85 chars | Medium (core label drifts on ~20% of runs) |
| Product description | 15 / 15 (100% unique) | 0% - zero exact matches | 1,954 chars | Low (specs and identity change every run) |
| Data extraction | 12 / 15 (80% unique) | 20% - only 3 runs matched another verbatim | 155 chars | High(correct facts in 14/15 runs, 1 schema failure) |


## 1. Sentiment Prompt — Failure Patterns

| # | Failure Pattern | Frequency | Example |
|---|---|---|---|
| 1 | Lexical drift — model swaps in a different (but semantically similar) second label instead of the "expected" one | 3/15 (Runs 4, 5, 8) | "positive feedback or a **customer compliment**"; "...a **satisfied customer review**"; "...a **positive review**" |
| 2 | Single-label collapse — model drops the secondary label entirely | 2/15 (Runs 10, 12) | "classified as **Positive Feedback**." (no second label) |
| 3 | Cosmetic formatting drift — bold vs. quoted labels, "This" vs. "The" as sentence opener | ~10/15 | Recurs throughout |
| 4 | Casing inconsistency | 2/15 (Runs 3, 7... lowercase "positive feedback") | "**positive feedback** or **customer satisfaction**" |

**Root cause:** No fixed label was specified in the prompt, so the model is free to generate its own secondary label each time.


## 2. Product Description Prompt — Failure Patterns

| # | Failure Pattern | Frequency | Example |
|---|---|---|---|
| 1 | Inconsistent product naming/branding | 15/15 (name changes almost every run) | "SwiftClick," "SwiftConnect," "Wireless Precision Mouse," "Wireless Ergonomic Mouse," "Wireless Comfort Mouse," "Wireless Freedom Mouse" |
| 2 | Contradictory technical specs across runs | High | DPI ranges vary: 800/1200/1600 (most runs) vs. up to 2400 (Run 12) vs. up to 3200 (10-run test, Run 6) |
| 3 | Contradictory mechanics | Moderate | Most runs say "single AA battery"; Run 6 says "single AA battery" but frames it as long-lasting rechargeable-style claims elsewhere; earlier 10-run test had explicit "rechargeable battery" contradicting AA claims |
| 4 | Structural inconsistency | Moderate | Most runs use header + bullet "Key Features" format; Run 6 uses a hybrid paragraph-then-bullet-summary structure not seen elsewhere |
| 5 | Fabricated features not grounded in the prompt | High | "quiet click technology," "adjustable DPI," "sleep mode," "33 feet range" — none specified in the original prompt, all invented, and inconsistently applied |

**Root cause:** The prompt supplies only two constraints (product = wireless mouse, price = $29.99). Every other detail — name, features, specs, structure — is generated freely, so the model has no anchor to stay consistent against. This is the only prompt with 0% exact-match consistency across all 15 runs.

---

## 3. Data Extraction Prompt — Failure Patterns

| # | Failure Pattern | Frequency | Example |
|---|---|---|---|
| 1 | Field-name inconsistency | ~5/15 | "Order Number" (most common) vs. "Order Item Number" (Run 5) vs. "Item Ordered" (Run 8) |
| 2 | Field renamed to a shorter variant | 3/15 (Runs 3, 6, 10) | "Delivery Speed" → "Delivery" |
| 3 | Punctuation inconsistency on the ID value | 1/15 (Run 14) | "#12345" → "12345" (# dropped) |
| 4 | Schema restructuring (most severe failure) | 1/15 (Run 10) | Header changed to "Customer Feedback Summary:"; "Packaging Condition" field was dropped and replaced with a merged "**Issue**: Damaged packaging" field |
| 5 | Preamble sentence variation | ~8/15 | "Here is the extracted information...", "Here are the extracted details...", or no preamble at all (direct bullet list) |

**Root cause:** No output schema (e.g., JSON with fixed keys) was specified, so while the model reliably extracts the *correct facts* (14/15 runs got all four values right), it's free to invent its own field names and structure each time. Run 10 is the standout failure: it didn't just relabel a field, it changed what fields existed — this would silently break any code parsing by key name.

---

## Cross-Prompt Failure

| Failure Type | Sentiment | Product Description | Extraction |
|---|---|---|---|
| Cosmetic wording/formatting drift | Yes (dominant) | Yes | Yes (dominant) |
| Content/fact drift | Yes (minor, 2 patterns) | Yes (severe, pervasive) | No (facts stayed correct) |
| Schema/structure drift | No | Yes (structure mostly stable, specs unstable) | Yes (1 severe case) |
| Fabrication / hallucination | No | Yes (high — invented specs) | No |
| Exact-match consistency | 20% | 0% | 20% |

## Overall Findings at 15 Runs

1. **More runs surfaced failure modes invisible at smaller sample sizes.** The sentiment prompt's "single-label collapse" (Runs 10, 12) never appeared in the 5-run or 10-run tests. The extraction prompt's schema-breaking Run 10 was also a first — a genuinely new, more severe failure category, not just more of the same wording noise.
2. **Consistency does not equal correctness, and vice versa.** The extraction prompt has the same exact-match rate as the sentiment prompt (12/15 unique) but is far more reliable at the fact level — because its answer space (facts in the source text) is narrow, while sentiment's secondary-label space is unconstrained.
3. **The product description prompt is the clear outlier** — 0% exact-match consistency across all 15 runs, and the only prompt where the model contradicts itself on factual-sounding claims (specs, battery type) between runs. This is a hallucination risk, not just a style-variation risk.
4. **Failure severity ranks:** Product description (fabricated, contradictory specs) > Sentiment (occasional label collapse/substitution) > Extraction (correct facts, but unstable field-naming that would break automated parsing).



# Part 3: Iteration 1 - Rewriting Simple Prompts
Objective: Improve prompts based on failure analysis by adding clarity, format requirements, and constraints.

### Step 6: Improve Sentiment Analysis Prompt

In [12]:
sentiment_prompt_v2 = """
Please analyze the sentiment of this customer's message "I love this product! It's exactly what I needed."
FORMAT:
1. You must classify the sentiment in one of the following terms: POSITIVE, NEGATIVE, or NEUTRAL.
2. Do not include any conversational filler, introductory text, markdown formatting, or punctuation.
CONSTRAINT: Use ONLY one of the classification word
"""
# Run the same prompt 15 times with temperature=0.7
num_runs=15
print("=== TESTING SENTIMENT PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(sentiment_prompt_v2, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

=== TESTING SENTIMENT PROMPT ===

Running 15 times...
Run 1: POSITIVE

Run 2: POSITIVE

Run 3: POSITIVE

Run 4: POSITIVE

Run 5: POSITIVE

Run 6: POSITIVE

Run 7: POSITIVE

Run 8: POSITIVE

Run 9: POSITIVE

Run 10: POSITIVE

Run 11: POSITIVE

Run 12: POSITIVE

Run 13: POSITIVE

Run 14: POSITIVE

Run 15: POSITIVE

📊 Analysis:
Total unique responses: 1 out of 15
Average length: 8 characters
--------------------------------------------------------------------------------


### Step 7: Improve Product Description Prompt

In [13]:
product_prompt_v2 = """
Write compelling product description for a wireless mouse that costs $29.99
Structure the descriptions as:
1. Use the title: **Tech Flow Wireless Mouse**
2. A short (3 lines) introductory paragraph
3. Highlight **Key Features** exactly titled and write 3 bullet points of features
Lenght Contraint: The entire product description must be under 120 words.
Style guideline: 
- Tone: Professional
- Keep it concise
- Do not exaggerate with phrases such as "greatest mouse ever"
- Do not use emoji
"""
num_runs=15
print("=== TESTING PRODUCT DESCRIPTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(product_prompt_v2, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)

=== TESTING PRODUCT DESCRIPTION PROMPT ===

Running 15 times...
Run 1: **Tech Flow Wireless Mouse**

Elevate your computing experience with the Tech Flow Wireless Mouse. Designed for precision and comfort, this mouse is perfect for both work and play, allowing you to navigate effortlessly through your tasks.

**Key Features**  
- **Ergonomic Design**: Contoured shape fits comfortably in your hand, reducing strain during extended use.  
- **Reliable Wireless Connection**: Enjoy a stable, lag-free connection with a range of up to 33 feet.  
- **Long Battery Life**: Equipped with energy-efficient technology, this mouse lasts up to 12 months on a single AA battery.

Run 2: **Tech Flow Wireless Mouse**

Experience seamless navigation with the Tech Flow Wireless Mouse, designed for efficiency and comfort. Perfect for both work and play, this mouse combines sleek design with reliable performance, making it an ideal companion for your laptop or desktop.

**Key Features**  
- **Reliable Wireles

### Step 8: Improve Data Extraction Prompt

In [14]:
extraction_prompt_v2="""
Extract relevant details from this customer feedback: 
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
1. OUTPUT FORMAT: JSON object
2. Include following fields in the output:
- "order_number" (include # symbol)
- "order_date"
- "delivery_speed"
- "package_condition"
3. Do not use any conversational text, introductory remarks, or explanations.
"""
num_runs
print("\n\n=== TESTING DATA EXTRACTION PROMPT ===")
print(f"\nRunning {num_runs} times...")
responses = test_prompt(extraction_prompt_v1, num_runs)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")
print("-" * 80)



=== TESTING DATA EXTRACTION PROMPT ===

Running 15 times...
Run 1: Here’s the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

Run 2: Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

Run 3: - **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

Run 4: Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

Run 5: - **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged

Run 6: Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March

## Comparision with Version 1

| Prompt | v1 Consistency | v2 Consistency | What fixed | What still needs work |
|---|---|---|---|---|
| Sentiment classification | 20% (12/15 unique) | 100% (1/15 unique) | Perfect Output | -- |
| Product description | 0% (15/15 unique) | High Structural Consistency | Branding, format, length, and style are locked in | Still hallucinating technical specs |
| Data extraction | 20% (12/15 unique) | 46% (8/15 unique) | Slight reduction in variations. Value extraction (e.g., #12345) was slightly more stable | 0% JSON compliance. Banned words failed; conversational intro text remained in 10/15 runs; key drift still occurred (Runs 11 & 12) |

# Part 4: Iteration 2 - Adding Structure & Constraints
Objective: Add few-shot examples, Chain-of-Thought reasoning, and advanced structure to dramatically improve consistency.